# 13 | Matched Subject-Level Cross-Validation: RF vs Two CNN Losses

**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**Aim:** Re-evaluate the Stage 12 weighted-loss improvement across three independent-participant GroupKFold partitions of the same bounded HARTH development cohort. Every fold fits Random Forest, Standard CNN and Weighted CNN on identical source windows and held-out participants. **The original two reserved Test IDs are never loaded.** This is exploratory development CV, not a final independent benchmark.

## 01. Imports and research components

All reusable source preparation, model fitting and evaluation logic live in Python modules. Code cells contain concise explanatory comments.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display

# Locate project root and register the research module paths.
root = Path.cwd().resolve()
while not (root / "code/python/src/core/data_loading.py").is_file():
    if root == root.parent: raise RuntimeError("Open this notebook inside the repository.")
    root = root.parent
sys.path.insert(0, str(root / "code/python/src"))
sys.path.insert(0, str(root / "studies/wearable_activity_recognition/src"))

from harth_data import HARTH_SUBJECTS, ACTIVITY_NAMES, SENSOR_COLUMNS, prepare_harth_window_cohort
from core.temporal_comparison import compare_models_group_cv
from core.group_cross_validation import compare_models_by_subject
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 02. Reconstruct the same bounded HARTH development cohort

Use the same official source recordings, subject IDs, 100-point strict-label windows and 500-per-subject label-independent cap used in Stages 10–12. Original Test participants are excluded before source acquisition.

In [ ]:
# Prepare all 29 development participants; protect both original Test IDs.
RESERVED_TEST = {"S023", "S032"}
DEVELOPMENT_SUBJECTS = sorted(set(HARTH_SUBJECTS) - RESERVED_TEST)
TASK_CLASSES = tuple(range(1, 9))
X, y, metadata, overview = prepare_harth_window_cohort(
    subjects=DEVELOPMENT_SUBJECTS,
    data_dir=root / "data/raw/harth_v2",
    task_classes=TASK_CLASSES, max_windows_per_subject=500,
    window_size=100, step_size=50, sampling_rate=50,
)
assert len(DEVELOPMENT_SUBJECTS) == len(set(DEVELOPMENT_SUBJECTS)) == 29
assert not (set(metadata["subject_id"]) & RESERVED_TEST)
print("Shared signal cohort:", X.shape)
display(overview[["subject_id", "eligible_windows", "sampled_windows"]])

## 03. Audit activity support before model fitting

Small classes are uneven across people. GroupKFold is subject-independent but does not stratify activity prevalence. Record the full participant-by-class table before comparing models.

In [ ]:
# Count source activity labels per development participant.
coverage = pd.crosstab(metadata["subject_id"], metadata["label"]).reindex(
    index=DEVELOPMENT_SUBJECTS, columns=TASK_CLASSES, fill_value=0
)
coverage.columns = [f"{code}: {ACTIVITY_NAMES[code]}" for code in TASK_CLASSES]
display(coverage)
print("Independent participants supporting each class:",
      (coverage > 0).sum(axis=0).to_dict())

## 04. Fit three matched models across independent-participant folds

Three folds are used to keep the first deep-learning CV experiment manageable on CPU. Each CNN is fitted from scratch with a fixed seed, with training-only normalisation and training-only class weights. Random Forest uses deterministic features from exactly the same windows.

In [ ]:
# Train all models on matching group-held-out development folds.
FOLDS = 3
EPOCHS = 5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
fold_results, subject_results, class_results, prediction_records = compare_models_group_cv(
    windows=X,
    labels=y,
    subject_ids=metadata["subject_id"].to_numpy(),
    classes=TASK_CLASSES,
    channel_names=SENSOR_COLUMNS,
    n_splits=FOLDS,
    epochs=EPOCHS,
    batch_size=128,
    learning_rate=1e-3,
    random_state=42,
    device=DEVICE,
    forbidden_subject_ids=RESERVED_TEST,
)
display(fold_results[[
    "fold", "model", "validation_subjects", "train_windows",
    "validation_windows", "validation_classes_present",
    "macro_f1", "balanced_accuracy"
]].round(4))

## 05. Compare fold-level results

The mean fold score is a descriptive summary. Fold scores are not independent replications, and a difference in scores is not by itself evidence of statistical superiority.

In [ ]:
# Summarise fixed-eight-class Macro F1 and Balanced Accuracy.
fold_summary = fold_results.groupby("model", as_index=False).agg(
    macro_f1_mean=("macro_f1", "mean"),
    macro_f1_sd=("macro_f1", "std"),
    balanced_accuracy_mean=("balanced_accuracy", "mean"),
)
display(fold_summary.round(4))

# Show each model's score in all three subject-held-out folds.
fig, ax = plt.subplots(figsize=(10, 5))
for name, table in fold_results.groupby("model", sort=False):
    table = table.sort_values("fold")
    ax.plot(table["fold"], table["macro_f1"], marker="o", label=name)
ax.set(title="HARTH: matched development GroupKFold Macro F1",
       xlabel="Held-out fold", ylabel="Macro F1", ylim=(0, 1),
       xticks=list(range(1, FOLDS + 1)))
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

## 06. Inspect pooled out-of-fold class performance

Every development window contributes exactly one held-out prediction per model. Pooled class F1 is a useful error diagnostic, but overlapping windows do not represent independent people.

In [ ]:
# Show class-wise metrics derived exclusively from held-out predictions.
class_results["activity"] = class_results["class"].map(
    lambda code: ACTIVITY_NAMES[int(code)]
)
display(class_results[[
    "model", "activity", "support", "precision", "recall", "f1"
]].round(4))

# Compare each class using grouped bars in the same Figure.
fig, ax = plt.subplots(figsize=(13, 5))
names = ["Random Forest", "Standard CNN", "Weighted CNN"]
positions = np.arange(len(TASK_CLASSES))
bar_width = 0.25
for i, name in enumerate(names):
    values = class_results.loc[class_results["model"] == name, "f1"].to_numpy()
    ax.bar(positions + (i-1)*bar_width, values, width=bar_width, label=name)
ax.set_xticks(positions, [ACTIVITY_NAMES[c] for c in TASK_CLASSES],
              rotation=35, ha="right")
ax.set(title="Out-of-fold activity F1 by model",
       ylabel="Pooled F1", ylim=(0, 1))
ax.grid(axis="y", alpha=0.2)
ax.legend()
fig.tight_layout()
plt.show()

## 07. Compare models for each held-out participant

Use fixed-eight-class Macro F1 per person, giving each participant an equal contribution to the paired summaries. Because some individuals have fewer than eight observed classes, report coverage separately.

In [ ]:
# Compare weighted against standard CNN and Random Forest on the same subjects.
display(subject_results[[
    "fold", "model", "subject_id", "windows",
    "observed_classes", "macro_f1_fixed_task"
]].sort_values(["subject_id", "model"]).round(4))
comparisons = [
    ("Weighted CNN", "Standard CNN"),
    ("Weighted CNN", "Random Forest"),
]
fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
for ax, (model_a, model_b) in zip(axes, comparisons):
    paired, pair_summary = compare_models_by_subject(
        subject_results, model_a=model_a, model_b=model_b,
        score_column="macro_f1_fixed_task",
    )
    display(pd.DataFrame([pair_summary]).round(4))
    ax.bar(paired["subject_id"], paired["difference_a_minus_b"])
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set(title=f"{model_a} minus {model_b}",
           ylabel="Paired fixed-task Macro F1")
    ax.grid(axis="y", alpha=0.2)
axes[-1].tick_params(axis="x", rotation=80)
axes[-1].set_xlabel("Held-out development participant")
fig.tight_layout()
plt.show()

## 08. Research quality gate

Check that every independent participant and every window is held out exactly once per model. This does not establish independent generalisation or replace a protected final Test evaluation.

In [ ]:
# Verify out-of-fold completeness and the protected original Test boundary.
assert set(subject_results["subject_id"]) == set(DEVELOPMENT_SUBJECTS)
assert not (set(subject_results["subject_id"]) & RESERVED_TEST)
assert len(fold_results) == 3 * FOLDS
assert len(subject_results) == 3 * len(DEVELOPMENT_SUBJECTS)
assert len(prediction_records) == 3 * len(X)
for name, table in prediction_records.groupby("model"):
    assert table["source_row"].nunique() == len(X)
    assert np.array_equal(
        table.sort_values("source_row")["true_label"].to_numpy(), y
    )
assert fold_results["macro_f1"].between(0, 1).all()
print("Matched GroupKFold, per-model source alignment: PASSED")
print("Original reserved Test loaded / evaluated: NO")
print("Independent confirmation of model-family benefit: NOT ESTABLISHED")